# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


In [16]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess


In [17]:
# Load API keys from the .env file in the project root

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI
Groq API Key not set (and this is optional)
OpenRouter API Key not set (and this is optional)


In [18]:
# Connect to client libraries.
# Gemini, Groq, Ollama and OpenRouter all expose an OpenAI-compatible endpoint, so we can reuse
# the OpenAI Python client for all of them - just pointing at a different base_url.

openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


In [19]:
# The open-source (and low-cost) models we'll compare as C++ code generators,
# and the client each one talks to (Ollama runs qwen2.5-coder/deepseek-coder-v2/gpt-oss:20b locally)

models = ["gemini-2.5-pro", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b", ]

clients = {"gemini-2.5-pro": gemini, "openai/gpt-oss-120b": groq, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [20]:
# Gather details about this machine (OS, CPU, compiler toolchain) so we can
# tell the LLM exactly what it's compiling for, and ask it to tune accordingly

from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '25.5.0',
  'version': 'Darwin Kernel Version 25.5.0: Tue Jun  9 22:18:58 PDT 2026; root:xnu-12377.121.10~1/RELEASE_ARM64_T6000',
  'kernel': '25.5.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin25.5.0'},
 'package_managers': ['xcode-select (CLT)', 'brew'],
 'cpu': {'brand': 'Apple M1 Max',
  'cores_logical': 10,
  'cores_physical': 10,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'g++': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'clang': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

In [21]:
# The compile/run commands for this machine, tuned for max performance with clang++

compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]


## And now, on with the main task

In [22]:
# The prompt that instructs a model to port Python -> fast C++.
# system_prompt sets the ground rules (C++ only, no prose);
# user_prompt_for() adds the machine details and the actual Python source to convert.

system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [23]:
# Build the standard system/user chat messages list for a porting request

def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [ ]:
# Save the generated C++ code to main.cpp so it can be compiled next

def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [25]:
# Send the porting request to the client library for the given model and return the C++ source.
# gpt-5 is the only model here that supports reasoning_effort, so it's added conditionally rather
# than sent (even as None) to every backend - Gemini/Groq/Ollama/OpenRouter would reject it.
# Markdown code fences are stripped since some models wrap their response in ```cpp ... ``` anyway.

def port(model, python):
    client = clients[model]
    kwargs = dict(model=model, messages=messages_for(python))
    if model.startswith('gpt-5'):
        kwargs["reasoning_effort"] = "high"
    response = client.chat.completions.create(**kwargs)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

In [26]:
# Sample workload: a deliberately slow, pure-Python numeric calculation.
# This is what we'll ask each LLM to port to fast C++, then benchmark against.

pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [27]:
# Run the raw Python code directly and capture its stdout, so the pi calculation's
# output can be returned/displayed instead of just printing straight to the notebook

def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [28]:
# Compile main.cpp (using the fast-compile flags above) and run the resulting binary
# three times, to see timing stabilize once warmed up

def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [29]:
# Gradio UI: paste/edit Python on the left, pick a model, and click "Convert code" to
# generate the C++ port on the right (this also writes the result to main.cpp via port())

with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [30]:
# Compile and run whatever C++ was last written to main.cpp by the UI above

compile_and_run()

Result: 3.141592656090
Execution Time: 0.035847 seconds

Result: 3.141592656090
Execution Time: 0.034469 seconds

Result: 3.141592656090
Execution Time: 0.048274 seconds



Traceback (most recent call last):
  File "/Users/robertkigobe/Documents/My_Research/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/robertkigobe/Documents/My_Research/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/robertkigobe/Documents/My_Research/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/robertkigobe/Documents/My_Research/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^

Qwen 2.5 Coder: Fail  
DeepSeek Coder v2: 0.114050084  
OpenAI gpt-oss 20B: 0.080438  
Qwen 30B: 0.113734  
OpenAI gpt-oss 120B: 1.407383




In Ed's experiments, the performance speedups were:

9th place: Qwen 2.5 Coder: Fail  
8th place: OpenAI GPT-OSS 120B: 14X speedup    
7th place: DeepSeek Coder v2: 168X speedup  
6th place: Qwen3 Coder 30B: 168X speedup   
5th place: Claude Sonnet 4.5: 184X speedup   
4th place: GPT-5: 233X speedup  
**3rd place: oss-20B: 238X speedup**  
2nd place: Grok 4: 1060X speedup  
1st place: Gemini 2.5 Pro: 1440X speedup  